# Early-fusion Residual-scSE U-Net: three seeds

Complete three-seed training and saved output. Original code retained.

Source: `graindiscoloration (2).ipynb`, zero-based cell 44.

Run in a fresh Kaggle GPU session with the appropriate dataset attached. Review the configuration near the top; original dataset/output paths are retained. Do not Run All across the repository.


In [ ]:
"""
Six-channel parameter-comparable early-fusion U-Net baseline.

Purpose
-------
Controlled baseline for the final RGB--multispectral GD segmentation model.
The six original channels [R, G, B, Green, Red-Edge, NIR] are concatenated
before a single Residual-scSE U-Net encoder. NDRE is not used.

The script runs seeds 42, 123, and 2026 sequentially. For every seed it:
  * trains from scratch on the provided 280/60/60 split;
  * selects the checkpoint using validation Dice only;
  * selects a no-TTA threshold using validation data only;
  * evaluates the test set once using that frozen threshold;
  * saves global and per-image metrics, the checkpoint, history, and config;
  * creates a downloadable ZIP immediately after the seed finishes.

After all seeds finish, it reports mean +/- sample standard deviation and
creates one combined ZIP. No test-time augmentation is used in this primary
controlled comparison.
"""

from __future__ import annotations

import csv
import json
import random
import shutil
import time
from pathlib import Path

import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


# =============================================================================
# 1. CONFIGURATION
# =============================================================================

# Set this manually only if automatic discovery fails.
# BASE_DIR = "/kaggle/input/datasets/hemal7/gd-rgbms-db"
# BASE_DIR = "/content/GD-RGBMS-DB"
BASE_DIR = None

WORK_DIR = (
    Path("/kaggle/working")
    if Path("/kaggle/working").exists()
    else Path("/content")
)
MASTER_OUT_DIR = WORK_DIR / "gd_6ch_early_fusion_unet_three_seeds"
MASTER_OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_SEEDS = [42, 123, 2026]
TARGET_SIZE = (896, 672)  # PIL convention: width, height
BATCH_SIZE = 2
ACCUMULATION_STEPS = 2
MAX_EPOCHS = 70
PATIENCE = 12
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0

PREFERRED_GN_GROUPS = 8
ENCODER_CHANNELS = [32, 64, 128, 248]
CACHE_TRAIN_AND_VALIDATION = True

# Save seed-42 probability maps for later qualitative and paired analysis.
# Add 123 and 2026 to save maps for every seed, at the cost of larger ZIPs.
SAVE_PROBABILITY_MAPS_FOR_SEEDS = {42}
SAVE_BINARY_PREDICTIONS_FOR_SEEDS = {42}
SHOW_DOWNLOAD_LINKS = True

MASK_POSITIVE_RGB = np.array([240, 128, 128], dtype=np.int16)

# Training-split statistics for the six original channels only.
CHANNEL_ORDER = ["R", "G", "B", "Green", "Red-Edge", "NIR"]
MEAN = np.array(
    [
        0.41819795966148376,
        0.46573567390441895,
        0.2516770660877228,
        0.21822915971279144,
        0.16206900775432587,
        0.23585167527198792,
    ],
    dtype=np.float32,
)
STD = np.array(
    [
        0.1540451943874359,
        0.1569177359342575,
        0.13375557959079742,
        0.07147330790758133,
        0.05124884843826294,
        0.04640943184494972,
    ],
    dtype=np.float32,
)

# Coarse thresholds are used during training to select the checkpoint.
# A finer search is performed after reloading the best checkpoint.
TRAIN_THRESHOLDS = np.round(np.arange(0.20, 0.86, 0.05), 2).tolist()
FINAL_THRESHOLDS = np.round(np.arange(0.20, 0.801, 0.01), 2).tolist()

try:
    BILINEAR = Image.Resampling.BILINEAR
    NEAREST = Image.Resampling.NEAREST
except AttributeError:
    BILINEAR = Image.BILINEAR
    NEAREST = Image.NEAREST


# =============================================================================
# 2. DATASET DISCOVERY AND REPRODUCIBILITY
# =============================================================================

def read_list(path: Path):
    with open(path, "r", encoding="utf-8") as file:
        return [line.strip() for line in file if line.strip()]


def valid_root(path: Path):
    required = [
        path / "RGB",
        path / "MS",
        path / "Mark",
        path / "train_list.txt",
        path / "val_list.txt",
        path / "test_list.txt",
    ]
    if not all(item.exists() for item in required):
        return False
    return (
        len(read_list(path / "train_list.txt")) == 280
        and len(read_list(path / "val_list.txt")) == 60
        and len(read_list(path / "test_list.txt")) == 60
    )


def find_dataset_root():
    roots = [Path("/kaggle/input"), Path("/content"), Path("/content/drive/MyDrive")]
    candidates = []
    for root in roots:
        if root.exists():
            for item in root.rglob("train_list.txt"):
                if valid_root(item.parent):
                    candidates.append(item.parent)
    if not candidates:
        raise FileNotFoundError(
            "The dataset with the provided 280/60/60 split was not found. "
            "Set BASE_DIR manually near the top of the script."
        )
    candidates.sort(key=lambda path: ("gd-rgbms" not in str(path).lower(), len(str(path))))
    return candidates[0]


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


if BASE_DIR is None:
    BASE_DIR = find_dataset_root()
BASE_DIR = Path(BASE_DIR)

train_names = read_list(BASE_DIR / "train_list.txt")
val_names = read_list(BASE_DIR / "val_list.txt")
test_names = read_list(BASE_DIR / "test_list.txt")

train_set, val_set, test_set = set(train_names), set(val_names), set(test_names)
if train_set & val_set or train_set & test_set or val_set & test_set:
    raise ValueError("Exact filename overlap exists across the dataset splits.")

rgb_dir = BASE_DIR / "RGB"
ms_dir = BASE_DIR / "MS"
mark_dir = BASE_DIR / "Mark"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = device.type == "cuda"

print("=" * 88)
print("SIX-CHANNEL EARLY-FUSION U-NET — THREE-SEED CONTROLLED BASELINE")
print("=" * 88)
print("Dataset       :", BASE_DIR)
print("Output        :", MASTER_OUT_DIR)
print("Device        :", device)
print("Split         :", len(train_names), len(val_names), len(test_names))
print("Input         :", TARGET_SIZE, "| channels:", CHANNEL_ORDER)
print("Seeds         :", RUN_SEEDS)
print("Primary mode  : no TTA")


# =============================================================================
# 3. PREPROCESSING AND RAM CACHE
# =============================================================================

def find_path(directory: Path, stem: str, extensions):
    for extension in extensions:
        path = directory / f"{stem}{extension}"
        if path.exists():
            return path
    raise FileNotFoundError(f"Could not find {stem} in {directory}")


def binary_mask(image: Image.Image):
    array = np.asarray(image.convert("RGB"), dtype=np.int16)
    colour_distance = np.abs(array - MASK_POSITIVE_RGB).sum(axis=2)
    return (colour_distance < 80).astype(np.float32)


def load_raw(name: str):
    stem = Path(name).stem
    rgb_path = find_path(rgb_dir, stem, [".JPG", ".jpg", ".JPEG", ".jpeg"])
    ms_path = find_path(ms_dir, stem, [".PNG", ".png"])
    mark_path = find_path(mark_dir, stem, [".png", ".PNG"])

    with Image.open(rgb_path) as item:
        rgb_image = item.convert("RGB").resize(TARGET_SIZE, BILINEAR)
    with Image.open(ms_path) as item:
        ms_image = item.convert("RGB").resize(TARGET_SIZE, BILINEAR)
    with Image.open(mark_path) as item:
        mark_image = item.convert("RGB").resize(TARGET_SIZE, NEAREST)

    rgb = np.asarray(rgb_image, dtype=np.float32).copy() / 255.0
    multispectral = np.asarray(ms_image, dtype=np.float32).copy() / 255.0
    mask = binary_mask(mark_image)
    image = np.concatenate([rgb, multispectral], axis=2)
    return image.astype(np.float32), mask, stem


RAM_CACHE = {}


def preload_cache(names):
    if not CACHE_TRAIN_AND_VALIDATION:
        return
    for name in tqdm(names, desc="Preloading train+validation cache"):
        image, mask, stem = load_raw(name)
        RAM_CACHE[name] = (image.astype(np.float16), mask.astype(np.uint8), stem)
    approximate_gb = sum(
        image.nbytes + mask.nbytes for image, mask, _ in RAM_CACHE.values()
    ) / (1024**3)
    print(f"RAM cache: {len(RAM_CACHE)} samples, approximately {approximate_gb:.2f} GB")


preload_cache(train_names + val_names)


class GDDataset(Dataset):
    def __init__(self, names, augment=False):
        self.names = list(names)
        self.augment = augment

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        name = self.names[index]
        if name in RAM_CACHE:
            cached_image, cached_mask, stem = RAM_CACHE[name]
            image = cached_image.astype(np.float32)
            mask = cached_mask.astype(np.float32)
        else:
            image, mask, stem = load_raw(name)

        if self.augment:
            if random.random() < 0.5:
                image = np.flip(image, axis=1).copy()
                mask = np.flip(mask, axis=1).copy()
            if random.random() < 0.5:
                image = np.flip(image, axis=0).copy()
                mask = np.flip(mask, axis=0).copy()
            if random.random() < 0.75:
                gain = np.random.uniform(0.88, 1.12, (1, 1, 3)).astype(np.float32)
                bias = np.random.uniform(-0.035, 0.035, (1, 1, 3)).astype(np.float32)
                image[..., :3] = np.clip(image[..., :3] * gain + bias, 0.0, 1.0)

        image = ((image - MEAN) / STD).astype(np.float32)
        return (
            torch.from_numpy(image.transpose(2, 0, 1)).float(),
            torch.from_numpy(mask[None]).float(),
            stem,
        )


def make_loaders(seed: int):
    generator = torch.Generator()
    generator.manual_seed(seed)
    train_loader = DataLoader(
        GDDataset(train_names, augment=True),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory,
        drop_last=False,
    )
    val_loader = DataLoader(
        GDDataset(val_names),
        batch_size=1,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory,
    )
    test_loader = DataLoader(
        GDDataset(test_names),
        batch_size=1,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory,
    )
    return train_loader, val_loader, test_loader


# =============================================================================
# 4. PARAMETER-COMPARABLE EARLY-FUSION RESIDUAL-scSE U-NET
# =============================================================================

def group_norm(channels: int):
    groups = min(PREFERRED_GN_GROUPS, channels)
    while channels % groups:
        groups -= 1
    return nn.GroupNorm(groups, channels, eps=1e-5, affine=True)


class SCSE(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        hidden = max(channels // reduction, 1)
        self.channel = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, hidden, 1),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden, channels, 1),
            nn.Sigmoid(),
        )
        self.spatial = nn.Sequential(nn.Conv2d(channels, 1, 1), nn.Sigmoid())

    def forward(self, x):
        return x * self.channel(x) + x * self.spatial(x)


class ResidualSCSEBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, dropout: float = 0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            group_norm(out_channels),
            nn.SiLU(inplace=True),
            nn.Dropout2d(dropout) if dropout else nn.Identity(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            group_norm(out_channels),
        )
        self.skip = (
            nn.Identity()
            if in_channels == out_channels
            else nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, bias=False),
                group_norm(out_channels),
            )
        )
        self.attention = SCSE(out_channels)

    def forward(self, x):
        residual = F.silu(self.block(x) + self.skip(x), inplace=True)
        return self.attention(residual)


class UpBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.conv = ResidualSCSEBlock(
            in_channels + skip_channels, out_channels, dropout
        )

    def forward(self, x, skip):
        x = F.interpolate(
            x, size=skip.shape[-2:], mode="bilinear", align_corners=False
        )
        return self.conv(torch.cat([x, skip], dim=1))


class EarlyFusionResidualSCSEUNet(nn.Module):
    def __init__(self, input_channels=6, channels=ENCODER_CHANNELS):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.pool = nn.MaxPool2d(2)
        self.encoder1 = ResidualSCSEBlock(input_channels, c1)
        self.encoder2 = ResidualSCSEBlock(c1, c2)
        self.encoder3 = ResidualSCSEBlock(c2, c3, 0.05)
        self.encoder4 = ResidualSCSEBlock(c3, c4, 0.10)
        self.center = ResidualSCSEBlock(c4, c4 * 2, 0.10)
        self.up4 = UpBlock(c4 * 2, c4, c4, 0.10)
        self.up3 = UpBlock(c4, c3, c3, 0.05)
        self.up2 = UpBlock(c3, c2, c2)
        self.up1 = UpBlock(c2, c1, c1)
        self.output = nn.Conv2d(c1, 1, 1)

    def forward(self, x):
        e1 = self.encoder1(x)
        e2 = self.encoder2(self.pool(e1))
        e3 = self.encoder3(self.pool(e2))
        e4 = self.encoder4(self.pool(e3))
        center = self.center(self.pool(e4))
        d4 = self.up4(center, e4)
        d3 = self.up3(d4, e3)
        d2 = self.up2(d3, e2)
        d1 = self.up1(d2, e1)
        return self.output(d1)


# =============================================================================
# 5. LOSS AND METRICS
# =============================================================================

class ComboLoss(nn.Module):
    def forward(self, logits, targets):
        bce_map = F.binary_cross_entropy_with_logits(
            logits, targets.float(), reduction="none"
        )
        bce = bce_map.mean()
        probabilities = torch.sigmoid(logits)
        pt = probabilities * targets + (1.0 - probabilities) * (1.0 - targets)
        focal = (bce_map * ((1.0 - pt) ** 2.0)).mean()
        dims = tuple(range(1, logits.ndim))
        intersection = (probabilities * targets).sum(dims)
        denominator = probabilities.sum(dims) + targets.sum(dims)
        dice_loss = 1.0 - (
            (2.0 * intersection + 1e-6) / (denominator + 1e-6)
        ).mean()
        return 0.35 * bce + 0.25 * focal + 0.40 * dice_loss


def confusion_tensor(probabilities, masks, threshold):
    prediction = probabilities >= threshold
    target = masks >= 0.5
    return [
        int((prediction & target).sum().item()),
        int((prediction & ~target).sum().item()),
        int((~prediction & target).sum().item()),
        int((~prediction & ~target).sum().item()),
    ]


def confusion_numpy(probability, mask, threshold):
    prediction = probability >= threshold
    target = mask >= 0.5
    return [
        int(np.logical_and(prediction, target).sum()),
        int(np.logical_and(prediction, ~target).sum()),
        int(np.logical_and(~prediction, target).sum()),
        int(np.logical_and(~prediction, ~target).sum()),
    ]


def metrics(values):
    tp, fp, fn, tn = values
    epsilon = 1e-7
    return {
        "dice": (2 * tp + epsilon) / (2 * tp + fp + fn + epsilon),
        "iou": (tp + epsilon) / (tp + fp + fn + epsilon),
        "precision": (tp + epsilon) / (tp + fp + epsilon),
        "recall": (tp + epsilon) / (tp + fn + epsilon),
        "accuracy": (tp + tn + epsilon) / (tp + fp + fn + tn + epsilon),
    }


def add_values(left, right):
    return [old + new for old, new in zip(left, right)]


def save_csv(path: Path, rows):
    if not rows:
        return
    with open(path, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def safe_load(path: Path):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


# =============================================================================
# 6. ONE COMPLETE SEED RUN
# =============================================================================

def run_seed(seed: int):
    seed_everything(seed)
    train_loader, val_loader, test_loader = make_loaders(seed)

    seed_dir = MASTER_OUT_DIR / f"seed_{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = seed_dir / f"best_6ch_early_fusion_seed{seed}.pt"

    model = EarlyFusionResidualSCSEUNet(input_channels=6).to(device)
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    expected_parameter_count = 7_902_787
    if parameter_count != expected_parameter_count:
        raise RuntimeError(
            f"Architecture mismatch: expected {expected_parameter_count:,} "
            f"parameters but found {parameter_count:,}."
        )

    criterion = ComboLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=4, min_lr=1e-6
    )
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

    print("\n" + "=" * 88)
    print(f"SEED {seed} | SIX-CHANNEL EARLY-FUSION U-NET")
    print("=" * 88)
    print("Parameters:", f"{parameter_count:,}")

    def train_epoch():
        model.train()
        optimizer.zero_grad(set_to_none=True)
        total_loss = 0.0
        for step, (images, masks, _) in enumerate(
            tqdm(train_loader, desc=f"Training seed {seed}", leave=False), start=1
        ):
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=device.type == "cuda"):
                logits = model(images)
                loss = criterion(logits, masks)
                scaled_loss = loss / ACCUMULATION_STEPS
            scaler.scale(scaled_loss).backward()
            total_loss += loss.item()
            if step % ACCUMULATION_STEPS == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
        return total_loss / len(train_loader)

    @torch.no_grad()
    def validate_training_checkpoint():
        model.eval()
        totals = {threshold: [0, 0, 0, 0] for threshold in TRAIN_THRESHOLDS}
        loss_total = 0.0
        for images, masks, _ in tqdm(
            val_loader, desc=f"Validation seed {seed}", leave=False
        ):
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=device.type == "cuda"):
                logits = model(images)
                loss_total += criterion(logits, masks).item()
            probabilities = torch.sigmoid(logits)
            for threshold in TRAIN_THRESHOLDS:
                totals[threshold] = add_values(
                    totals[threshold],
                    confusion_tensor(probabilities, masks, threshold),
                )

        rows = []
        for threshold, values in totals.items():
            row = metrics(values)
            row.update(
                {
                    "threshold": float(threshold),
                    "loss": loss_total / len(val_loader),
                }
            )
            rows.append(row)
        return max(rows, key=lambda item: item["dice"])

    history = []
    best_dice = -1.0
    best_epoch = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        start = time.time()
        train_loss = train_epoch()
        validation = validate_training_checkpoint()
        scheduler.step(validation["dice"])
        current_lr = optimizer.param_groups[0]["lr"]
        minutes = (time.time() - start) / 60.0

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": validation["loss"],
            "val_dice": validation["dice"],
            "val_iou": validation["iou"],
            "val_precision": validation["precision"],
            "val_recall": validation["recall"],
            "val_accuracy": validation["accuracy"],
            "val_threshold": validation["threshold"],
            "learning_rate": current_lr,
            "minutes": minutes,
        }
        history.append(row)

        print(
            f"Seed {seed} | Epoch {epoch:03d} | train {train_loss:.4f} | "
            f"val {validation['loss']:.4f} | Dice {validation['dice']:.4f} | "
            f"IoU {validation['iou']:.4f} | P {validation['precision']:.4f} | "
            f"R {validation['recall']:.4f} | th {validation['threshold']:.2f} | "
            f"lr {current_lr:.2e} | {minutes:.1f}m"
        )

        if validation["dice"] > best_dice:
            best_dice = validation["dice"]
            best_epoch = epoch
            torch.save(
                {
                    "model": model.state_dict(),
                    "epoch": epoch,
                    "best_training_validation_dice": best_dice,
                    "training_validation_threshold": validation["threshold"],
                    "seed": seed,
                    "architecture": "EarlyFusionResidualSCSEUNet",
                    "input_channels": CHANNEL_ORDER,
                    "target_size": TARGET_SIZE,
                    "mean": MEAN.tolist(),
                    "std": STD.tolist(),
                    "normalization": "GroupNorm",
                    "encoder_channels": ENCODER_CHANNELS,
                    "parameter_count": parameter_count,
                },
                checkpoint_path,
            )
            print("Saved:", checkpoint_path)

        if epoch - best_epoch >= PATIENCE:
            print(f"Early stopping: {PATIENCE} consecutive non-improving epochs.")
            break

    save_csv(seed_dir / "training_history.csv", history)

    checkpoint = safe_load(checkpoint_path)
    model.load_state_dict(checkpoint["model"], strict=True)
    model.eval()

    @torch.no_grad()
    def collect_records(loader, description):
        records = []
        for images, masks, stems in tqdm(loader, desc=description):
            images = images.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=device.type == "cuda"):
                probabilities = torch.sigmoid(model(images))
            records.append(
                {
                    "stem": stems[0],
                    "mask": masks[0, 0].numpy().astype(np.float32),
                    "probability": probabilities[0, 0]
                    .float()
                    .cpu()
                    .numpy()
                    .astype(np.float32),
                }
            )
        return records

    def threshold_search(records):
        rows = []
        for threshold in FINAL_THRESHOLDS:
            total = [0, 0, 0, 0]
            for record in records:
                total = add_values(
                    total,
                    confusion_numpy(record["probability"], record["mask"], threshold),
                )
            row = metrics(total)
            row["threshold"] = float(threshold)
            rows.append(row)
        return max(rows, key=lambda item: item["dice"]), rows

    def evaluate(records, threshold):
        total = [0, 0, 0, 0]
        per_image = []
        for record in records:
            values = confusion_numpy(record["probability"], record["mask"], threshold)
            total = add_values(total, values)
            row = metrics(values)
            target = record["mask"] >= 0.5
            prediction = record["probability"] >= threshold
            row.update(
                {
                    "stem": record["stem"],
                    "threshold": float(threshold),
                    "gt_coverage_percent": float(target.mean() * 100.0),
                    "predicted_coverage_percent": float(prediction.mean() * 100.0),
                    "soft_coverage_percent": float(record["probability"].mean() * 100.0),
                    "tp": values[0],
                    "fp": values[1],
                    "fn": values[2],
                    "tn": values[3],
                }
            )
            per_image.append(row)
        result = metrics(total)
        result.update(
            {
                "seed": seed,
                "tta_mode": "no_tta",
                "threshold": float(threshold),
                "tp": total[0],
                "fp": total[1],
                "fn": total[2],
                "tn": total[3],
            }
        )
        return result, per_image

    validation_records = collect_records(
        val_loader, f"Fine validation threshold search seed {seed}"
    )
    validation_best, threshold_rows = threshold_search(validation_records)
    save_csv(seed_dir / "validation_threshold_search_no_tta.csv", threshold_rows)
    selected_threshold = validation_best["threshold"]

    # The test set is evaluated only after the validation threshold is frozen.
    test_records = collect_records(test_loader, f"Final no-TTA test seed {seed}")
    test_result, per_image = evaluate(test_records, selected_threshold)
    save_csv(seed_dir / "primary_no_tta_test_metrics.csv", [test_result])
    save_csv(seed_dir / "per_image_test_metrics_no_tta.csv", per_image)

    if seed in SAVE_PROBABILITY_MAPS_FOR_SEEDS:
        probability_dir = seed_dir / "test_probability_uint16"
        probability_dir.mkdir(parents=True, exist_ok=True)
        for record in tqdm(test_records, desc=f"Saving probability maps seed {seed}"):
            probability_uint16 = np.round(
                np.clip(record["probability"], 0.0, 1.0) * 65535.0
            ).astype(np.uint16)
            Image.fromarray(probability_uint16).save(
                probability_dir / f"{record['stem']}.png"
            )

    if seed in SAVE_BINARY_PREDICTIONS_FOR_SEEDS:
        prediction_dir = seed_dir / "test_binary_predictions"
        prediction_dir.mkdir(parents=True, exist_ok=True)
        for record in tqdm(test_records, desc=f"Saving predictions seed {seed}"):
            prediction = (record["probability"] >= selected_threshold).astype(np.uint8) * 255
            Image.fromarray(prediction).save(prediction_dir / f"{record['stem']}.png")

    configuration = {
        "experiment": "Six-channel parameter-comparable early-fusion U-Net baseline",
        "architecture": "EarlyFusionResidualSCSEUNet",
        "input_channel_order": CHANNEL_ORDER,
        "input_channel_count": 6,
        "derived_ndre_included": False,
        "fusion_type": "Input-level concatenation before one encoder",
        "modality_specific_branches": False,
        "spatial_gates": False,
        "cross_stage_feedback": False,
        "auxiliary_heads": False,
        "dataset_root": str(BASE_DIR),
        "seed": seed,
        "split_sizes": {
            "train": len(train_names),
            "validation": len(val_names),
            "test": len(test_names),
        },
        "target_size_width_height": list(TARGET_SIZE),
        "batch_size": BATCH_SIZE,
        "accumulation_steps": ACCUMULATION_STEPS,
        "effective_batch_size": BATCH_SIZE * ACCUMULATION_STEPS,
        "epochs_requested": MAX_EPOCHS,
        "epochs_completed": len(history),
        "best_epoch": checkpoint["epoch"],
        "early_stopping_patience": PATIENCE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "scheduler": "ReduceLROnPlateau(mode=max, factor=0.5, patience=4, min_lr=1e-6)",
        "normalization": "GroupNorm",
        "preferred_group_count": PREFERRED_GN_GROUPS,
        "encoder_channels": ENCODER_CHANNELS,
        "parameter_count": parameter_count,
        "reference_final_model_deployment_parameters": 7_950_033,
        "parameter_difference": parameter_count - 7_950_033,
        "parameter_difference_percent": 100.0 * (parameter_count - 7_950_033) / 7_950_033,
        "loss": "0.35 BCE + 0.25 focal(gamma=2) + 0.40 Dice",
        "mean": MEAN.tolist(),
        "std": STD.tolist(),
        "normalization_statistics_source": "Training split only",
        "training_initialization": "From scratch",
        "external_pretrained_weights": False,
        "checkpoint_selection": "Highest validation Dice only",
        "inference_mode": "No TTA",
        "threshold_selection": "Fine validation search only",
        "selected_validation_threshold": selected_threshold,
        "validation_metrics_at_selected_threshold": validation_best,
        "primary_no_tta_test_metrics": test_result,
        "test_use_policy": "Test data did not select the checkpoint or threshold.",
    }
    with open(seed_dir / "experiment_configuration.json", "w", encoding="utf-8") as file:
        json.dump(configuration, file, indent=2)

    checkpoint["selected_validation_threshold"] = selected_threshold
    checkpoint["fine_validation_metrics"] = validation_best
    checkpoint["primary_no_tta_test_metrics"] = test_result
    torch.save(checkpoint, checkpoint_path)

    seed_archive = shutil.make_archive(
        str(WORK_DIR / f"gd_6ch_early_fusion_unet_seed{seed}_results"),
        "zip",
        str(seed_dir),
    )

    print("\n" + "-" * 88)
    print(f"SEED {seed} FINISHED")
    print("Best epoch       :", checkpoint["epoch"])
    print("Val threshold    :", f"{selected_threshold:.2f}")
    print("Test Dice        :", f"{test_result['dice']:.6f}")
    print("Test IoU         :", f"{test_result['iou']:.6f}")
    print("Test Precision   :", f"{test_result['precision']:.6f}")
    print("Test Recall      :", f"{test_result['recall']:.6f}")
    print("Downloadable ZIP :", seed_archive)

    if SHOW_DOWNLOAD_LINKS:
        try:
            from IPython.display import FileLink, display

            display(FileLink(seed_archive))
        except Exception as error:
            print("Inline download link could not be displayed:", error)

    del model, optimizer, scheduler, scaler
    del validation_records, test_records
    if device.type == "cuda":
        torch.cuda.empty_cache()

    return test_result


# =============================================================================
# 7. RUN ALL SEEDS AND REPORT MEAN +/- SAMPLE SD
# =============================================================================

all_results = []
for run_seed_value in RUN_SEEDS:
    all_results.append(run_seed(run_seed_value))

save_csv(MASTER_OUT_DIR / "three_seed_individual_results.csv", all_results)

summary_rows = []
for metric_name in ["dice", "iou", "precision", "recall", "accuracy"]:
    values = np.array([row[metric_name] for row in all_results], dtype=np.float64)
    summary_rows.append(
        {
            "metric": metric_name,
            "mean": float(values.mean()),
            "sample_standard_deviation": float(values.std(ddof=1)),
            "minimum": float(values.min()),
            "maximum": float(values.max()),
            "number_of_seeds": len(values),
            "seeds": ",".join(str(seed) for seed in RUN_SEEDS),
        }
    )

save_csv(MASTER_OUT_DIR / "three_seed_mean_std_summary.csv", summary_rows)

combined_archive = shutil.make_archive(
    str(WORK_DIR / "gd_6ch_early_fusion_unet_three_seed_results"),
    "zip",
    str(MASTER_OUT_DIR),
)

print("\n" + "=" * 88)
print("THREE-SEED SIX-CHANNEL EARLY-FUSION BASELINE FINISHED")
print("=" * 88)
for row in summary_rows:
    print(
        f"{row['metric']:10s}: {row['mean']:.6f} +/- "
        f"{row['sample_standard_deviation']:.6f}"
    )
print("Combined ZIP:", combined_archive)
print("\nDownload every seed ZIP after it finishes; do not wait for the Kaggle session to end.")

if SHOW_DOWNLOAD_LINKS:
    try:
        from IPython.display import FileLink, display

        display(FileLink(combined_archive))
    except Exception as error:
        print("Inline download link could not be displayed:", error)
